## Load segments from Slicer

In [1]:
import numpy
import sys
import nrrd
from scipy import ndimage
import tomllib
import os
import shutil

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import tetra_mesh_from_stl, tetra_shell_from_two_surfaces, set_the_offset, getSurfaceMesh
from image_processing_functions import load_vti


with open("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/config.toml", mode="rb") as f:
    config = tomllib.load(f)

sequence = "T1"
mask_type = "nrrd"

if sequence == "T2":
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file_T2"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename_T2"]
    segment_folder  = config["data"]["segment_folder_T2"]
else:
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename"]
    segment_folder  = config["data"]["segment_folder"]

if mask_type == "vti":
    output_folder = output_folder[:-1]+"_vti/"

export_name = base + output_folder + segment_filename[:-8]


## Load segmentation mask

In [2]:
data, header = nrrd.read(base + segment_folder + segment_filename)

dirs = [numpy.array(d, dtype=float) for d in header["space directions"] if d is not None and not numpy.any(numpy.isnan(numpy.array(d, dtype=float)))]
dirs = numpy.array(dirs)

pixel_spacing = numpy.linalg.norm(dirs, axis=1)           # mm per voxel, in array axis order
direction = (dirs / pixel_spacing[:, None]).T          # columns = unit axis directions
origin = numpy.array(header["space origin"], dtype=float)

print("original pixel spacing = ", pixel_spacing)
print("segmentation origin = ", origin)
# numpy.savetxt(base + output_folder + "segmentation_origin_"+sequence+".txt", origin)
    
if mask_type == "vti":
    input_file = base + "image/mask_warped.vti"
    data, _, origin = load_vti(input_file)


original pixel spacing =  [0.3125     0.3125     0.50000006]
segmentation origin =  [-55.95619965 -90.20069885  20.49090004]


In [3]:
segments = []
i = 0
while f"Segment{i}_Name" in header:
    name  = header[f"Segment{i}_Name"]
    label = int(header.get(f"Segment{i}_LabelValue", i + 1))
    layer = int(header.get(f"Segment{i}_Layer", 0))
    vol   = data[layer] if data.ndim == 4 else data
    segments.append((name, vol == label))
    print(f"{name}: layer={layer}, value={label}, voxels={segments[-1][1].sum()}")
    i += 1

combined = numpy.zeros(segments[0][1].shape, dtype=numpy.uint8)
for new_id, (name, mask) in enumerate(segments, start=1):
    combined[mask] = new_id          # later segments overwrite earlier ones where they overlap

data = combined
id_max = int(data.max())
print(data.shape)


print()
print("original dimensions = ", data.shape)
print("max segment ID = ", id_max)
print()
print("Unique IDs")
with open(base + output_folder + "segment_IDs" + "_" + sequence + ".txt", "w") as f:
    for new_id, (name, _) in enumerate(segments, start=1):
        f.write(f"{name}\t{new_id}\n")
        print(f"{name}: new ID = {new_id}")

optical nerve: layer=0, value=1, voxels=15624
eyeball: layer=0, value=3, voxels=160561
lateral rectus: layer=0, value=4, voxels=14436
medial rectus: layer=0, value=5, voxels=13164
inferior rectus: layer=0, value=6, voxels=11961
superior rectus: layer=0, value=7, voxels=13568
lens: layer=0, value=2, voxels=1508
(145, 209, 101)

original dimensions =  (145, 209, 101)
max segment ID =  7

Unique IDs
optical nerve: new ID = 1
eyeball: new ID = 2
lateral rectus: new ID = 3
medial rectus: new ID = 4
inferior rectus: new ID = 5
superior rectus: new ID = 6
lens: new ID = 7


In [4]:
pixel_spacing = numpy.asarray(pixel_spacing, dtype=float)
target = pixel_spacing.min()

if not numpy.allclose(pixel_spacing, target):
    factors = pixel_spacing / target
    data = ndimage.zoom(data, factors, order=0)
    print("resampling factors = ", factors)
    print("new dimensions = ", data.shape)

pixel_spacing = target

print("Final uniform pixel_spacing = ", pixel_spacing)

resampling factors =  [1.         1.         1.60000019]
new dimensions =  (145, 209, 162)
Final uniform pixel_spacing =  0.3124999999999999


In [5]:
eyeball_id = 2
ignored_ids = [-1]
sclera_id = -1

smooth_ids = range(1,numpy.max(data)+1)

## Stage I
### Intial surface mesh

In [6]:
for id in range(1,numpy.max(data)+1):
    if id not in ignored_ids:

        print("id = ", id)

        if len(numpy.where((data>id-0.5) & (data<id+0.5 ))[0])>0:

            mask = numpy.zeros(data.shape, dtype=numpy.uint8)
            mask[(data>id-0.5) & (data<id+0.5 )]=1

            if id == sclera_id:
                mask[(data>eyeball_id-0.5) & (data<eyeball_id+0.5 )]=1
                
            print(len(numpy.where(mask==1)[0]))

            if id in smooth_ids:
                mask = ndimage.median_filter(mask, size=2)
            mask[mask<0.5]=0
            mask[mask>=0.5]=1



            numpy.save(export_name +"_filtered_surf_id_"+str(id), mask)

            if id ==eyeball_id:
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing, True, "in" )

                mask_voxels = numpy.argwhere(mask == 1)  # shape (N, 3), coordinates in (z, y, x)
                # compute mean voxel position
                mesh_offset = mask_voxels.mean(axis=0)*pixel_spacing  
                numpy.savetxt(base + output_folder + "mesh_offset_"+sequence+".txt", mesh_offset)

            else:
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing, False )
            print()


id =  1
25127
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_1.stl

id =  2
258812
Problem edges: 0
Outer and inner surfaces written.

id =  3
23220
Problem edges: 2
Open Hole Edges: 0
Non-Manifold Junctions: 2
 -> Remaining Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_3.stl

id =  4
21124
Problem edges: 2
Open Hole Edges: 0
Non-Manifold Junctions: 2
 -> Remaining Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_4.stl

id =  5
19237
Problem edges: 2
Open Hole Edges: 0
Non-Manifold Junctions: 2
 -> Remaining Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_5.stl

id =  6
21758
Problem edges: 1
Open Hole Edges: 0
Non-Manifold Junctions: 1
 -> R

## Stage II
### Re-mesh with given resolution

In [7]:
h=0.4

out_folder_centered = "re-meshed_h_"+str(h).replace(".","-")+"_cenetered_"+sequence+"/"
out_folder = "re-meshed_h_"+str(h).replace(".","-")+"_"+sequence+"/"

for folder in [out_folder_centered, out_folder]:
    if not os.path.isdir(base + output_folder + folder):
        os.mkdir(base + output_folder + folder)



for id in range(1,numpy.max(data)+1):
    if (id !=eyeball_id) and (id not in ignored_ids):
        stl_file = export_name +"_surf_id_"+str(id)+".stl"

        if os.path.isfile(stl_file):
            tetra_mesh_from_stl(stl_file, base  + output_folder + out_folder_centered + "id_"+str(id)+"_" + "_h_"+str(h).replace(".","-"), h)
            shutil.copy(base  + output_folder + out_folder_centered + "id_"+str(id)+"_" + "_h_"+str(h).replace(".","-") +".vtk", base  + output_folder + out_folder + "id_"+str(id)+"_" + "_h_"+str(h).replace(".","-")+".vtk")
            shutil.copy(base  + output_folder + out_folder_centered + "id_"+str(id)+"_" + "_h_"+str(h).replace(".","-") +".stl", base  + output_folder + out_folder + "id_"+str(id)+"_" + "_h_"+str(h).replace(".","-")+".stl")

            set_the_offset(base + output_folder + out_folder_centered + "id_"+str(id)+"_" + "_h_"+str(h).replace(".","-"), mesh_offset)


Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_1.stl'...
Info    : 19830 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_1.stl'
Info    : Classifying surfaces (angle: 89)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 19830 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 9867 triangles split in 2 parts because parametrized triangles are too small (8.29431e-09)
Info    :  - Level 2 partition with 4891 triangles split in 2 parts because parametrized triangles are too small (9.78587e-09)
Info    : Found 4 model surfaces
Info    : Found 3 model curves
Info    : Done classifying surfaces (Wall 0.303943s, CPU 0.290414s)
Info    : Creating geometry of discrete curves...
I

Warning: STL can only write triangle cells. Discarding line, vertex.

Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_3.stl'...
Info    : 22732 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_3.stl'
Info    : Classifying surfaces (angle: 89)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 22732 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 11496 triangles split in 2 parts because parametrized triangles are too small (9.45684e-09)
Info    : Found 3 model surfaces
Info    : Found 2 model curves
Info    : Done classifying surfaces (Wall 0.315129s, CPU 0.29506s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 1.1459e-05s, CPU 1.1e-05s)
Info    : Creating geometry of discrete sur

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_4.stl'...
Info    : 17908 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_4.stl'
Info    : Classifying surfaces (angle: 89)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 17908 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.167201s, CPU 0.164572s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 6.70878e-06s, CPU 6e-06s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 0.0682937s, CPU 0.066289s)                            
Info

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

... (2 outputs left out)

Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_6.stl'...
Info    : 18100 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_6.stl'
Info    : Classifying surfaces (angle: 89)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 18100 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.174298s, CPU 0.171558s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 7.6252e-06s, CPU 7e-06s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 0.0695956s, CPU 0.066845s)                            
Info 

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_7.stl'...
Info    : 3284 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_7.stl'
Info    : Classifying surfaces (angle: 89)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 3284 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.0298628s, CPU 0.029221s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 5.62472e-06s, CPU 5e-06s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 0.0119675s, CPU 0.011357s)                            
Info 

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

In [8]:

stl_file = export_name +"_surf_id_"+str(eyeball_id)+".stl"


tetra_shell_from_two_surfaces(stl_file.replace(".stl", "_outer.stl"), stl_file.replace(".stl", "_inner.stl"), base + output_folder + out_folder_centered + "id_"+str(eyeball_id)+"_shell_" + "_h_"+str(h).replace(".","-"), h)
shutil.copy(base  + output_folder + out_folder_centered + "id_"+str(eyeball_id).replace(".","-")+"_shell_" + "_h_"+str(h).replace(".","-") +".vtk", base  + output_folder + out_folder + "id_"+str(eyeball_id).replace(".","-")+"_shell_" + "_h_"+str(h).replace(".","-") +".vtk")
shutil.copy(base  + output_folder + out_folder_centered + "id_"+str(eyeball_id).replace(".","-")+"_shell_" + "_h_"+str(h).replace(".","-") +".stl", base  + output_folder + out_folder + "id_"+str(eyeball_id).replace(".","-")+"_shell_" + "_h_"+str(h).replace(".","-") +".stl")

set_the_offset(base  + output_folder + out_folder_centered + "id_"+str(eyeball_id).replace(".","-")+"_shell_" + "_h_"+str(h).replace(".","-"), mesh_offset)


Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_2_outer.stl'...
Info    : 61116 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_2_outer.stl'
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_2_inner.stl'...
Info    : 61116 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh/Segmentat_surf_id_2_inner.stl'
Info    : Classifying surfaces (angle: 40)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 61116 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 0 partition with 61116 triangles split in 2 parts bec

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.